In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# Setup

In [2]:
import subprocess
subprocess.run([
    'pip', 'install', 'mlflow', 'dagshub', 'scikit-learn', 'pandas',
    'matplotlib', 'seaborn', 'imbalanced-learn', 'xgboost', '--quiet'
])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/8

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.


CompletedProcess(args=['pip', 'install', 'mlflow', 'dagshub', 'scikit-learn', 'pandas', 'matplotlib', 'seaborn', 'imbalanced-learn', 'xgboost', '--quiet'], returncode=0)

In [3]:
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TARGET = 'isFraud'
ID_COL = 'TransactionID'
TIME_COL = 'TransactionDT'

In [4]:
from kaggle_secrets import UserSecretsClient
import dagshub
import mlflow
import mlflow.sklearn

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret('DAGSHUB_TOKEN')

dagshub.auth.add_app_token(token=dagshub_token)
dagshub.init(repo_owner='ngval22', repo_name='fraud-detection-classification', mlflow=True)
print('Tracking URI:', mlflow.get_tracking_uri())

EXPERIMENT_NAME = 'XGBoost_Training'
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name='XGB_Parent') as parent_run:
    PARENT_RUN_ID = parent_run.info.run_id
    mlflow.log_param('model_family', 'XGBoost')
    mlflow.log_param('random_state', RANDOM_STATE)
    mlflow.log_param('experiment', EXPERIMENT_NAME)

print('Parent run ID:', PARENT_RUN_ID)

Accessing as ngval22

Initialized MLflow to track repo "ngval22/fraud-detection-classification"

Repository ngval22/fraud-detection-classification initialized!

Tracking URI: https://dagshub.com/ngval22/fraud-detection-classification.mlflow
🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
Parent run ID: 191ca1f68a9e404eaea3a0f32d9f031e


In [5]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train_merged = train_transaction.merge(train_identity, on=ID_COL, how='left')
test_merged  = test_transaction.merge(test_identity,  on=ID_COL, how='left')

y_train     = train_merged[TARGET].copy()
X_raw_train = train_merged.drop(columns=[TARGET])
X_raw_test  = test_merged.copy()
test_ids    = test_transaction[ID_COL].copy()

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
BASE_SCALE_POS_WEIGHT = neg_count / pos_count

print('train merged:', train_merged.shape)
print('test merged: ', test_merged.shape)
print(f'Fraud rate: {y_train.mean():.4f}')
print(f'XGBoost base scale_pos_weight: {BASE_SCALE_POS_WEIGHT:.2f}')

train merged: (590540, 434)
test merged:  (506691, 433)
Fraud rate: 0.0350
XGBoost base scale_pos_weight: 27.58


# Cleaning

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier


class CleaningTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, missing_thresh=0.50, id_col=ID_COL,
                 time_col=TIME_COL, target=TARGET):
        self.missing_thresh = missing_thresh
        self.id_col = id_col
        self.time_col = time_col
        self.target = target

    def fit(self, X, y=None):
        missing_pct = X.isnull().mean()
        self.cols_to_drop_ = [
            c for c in missing_pct[missing_pct > self.missing_thresh].index
            if c not in [self.target, self.id_col]
        ]
        self.common_cols_ = [
            c for c in X.columns
            if c not in self.cols_to_drop_
            and c not in [self.target, self.id_col]
        ]
        return self

    def transform(self, X):
        out = X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns],
                     errors='ignore')
        out = out.drop(columns=[self.id_col], errors='ignore')
        out = out.drop(columns=[self.target], errors='ignore')
        return out[[c for c in self.common_cols_ if c in out.columns]]


cleaning_probe = CleaningTransformer().fit(X_raw_train, y_train)
print('Dropped columns:', len(cleaning_probe.cols_to_drop_))
print('Kept columns:', len(cleaning_probe.common_cols_))

Dropped columns: 214
Kept columns: 218


# Feature Engineering

In [7]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    IMPORTANT_NUM = ['D1','D2','D3','D4','D5','D6',
                     'C1','C2','C3','C4','C5',
                     'M1','M2','M3','M4','M5']

    def fit(self, X, y=None):
        self.cat_cols_ = X.select_dtypes(include='object').columns.tolist()
        self.ord_enc_ = OrdinalEncoder(
            handle_unknown='use_encoded_value', unknown_value=-1
        )
        if self.cat_cols_:
            self.ord_enc_.fit(X[self.cat_cols_].fillna('__MISSING__'))
        self.missing_flag_cols_ = [c for c in self.IMPORTANT_NUM if c in X.columns]
        return self

    def transform(self, X):
        out = X.copy()

        if 'TransactionAmt' in out.columns:
            out['TransactionAmt_log'] = np.log1p(out['TransactionAmt'])
            out['TransactionAmt_cents'] = (out['TransactionAmt'] % 1).round(2)
            out['Amt_is_round'] = (out['TransactionAmt'] % 1 == 0).astype(int)

        if TIME_COL in out.columns:
            out['hour'] = (out[TIME_COL] // 3600) % 24
            out['day_of_week'] = (out[TIME_COL] // (3600 * 24)) % 7
            out['is_weekend'] = out['day_of_week'].isin([5, 6]).astype(int)
            out = out.drop(columns=[TIME_COL])

        for col in self.missing_flag_cols_:
            out[f'{col}_missing'] = out[col].isnull().astype(int)

        if self.cat_cols_:
            out[self.cat_cols_] = self.ord_enc_.transform(
                out[self.cat_cols_].fillna('__MISSING__')
            )

        return out


fe_probe = FeatureEngineeringTransformer().fit(cleaning_probe.transform(X_raw_train.head(5000)), y_train.head(5000))
print('Categorical columns encoded:', len(fe_probe.cat_cols_))
print('Missing indicator columns:', len(fe_probe.missing_flag_cols_))

Categorical columns encoded: 9
Missing indicator columns: 13


# Feature Selection

In [8]:
class XGBFeatureSelectionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, corr_thresh=0.98, var_thresh=0.0,
                 n_features=80, sample_size=80000,
                 random_state=RANDOM_STATE):
        self.corr_thresh = corr_thresh
        self.var_thresh = var_thresh
        self.n_features = n_features
        self.sample_size = sample_size
        self.random_state = random_state

    def fit(self, X, y=None):
        self.input_columns_ = X.columns.tolist()

        imp = SimpleImputer(strategy='median')
        X_imp = imp.fit_transform(X)
        X_imp_df = pd.DataFrame(X_imp, columns=self.input_columns_)

        corr = X_imp_df.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        self.cols_drop_corr_ = [c for c in upper.columns if any(upper[c] > self.corr_thresh)]
        surviving = [c for c in self.input_columns_ if c not in self.cols_drop_corr_]

        vt = VarianceThreshold(threshold=self.var_thresh)
        vt.fit(X_imp_df[surviving])
        self.cols_keep_vt_ = [f for f, keep in zip(surviving, vt.get_support()) if keep]

        y_series = pd.Series(y).reset_index(drop=True)
        if len(y_series) > self.sample_size:
            per_class_n = min(y_series.value_counts().min(), self.sample_size // 2)
            sample_idx = y_series.groupby(y_series).sample(
                n=per_class_n,
                random_state=self.random_state
            ).index
            X_sample = X_imp_df.iloc[sample_idx][self.cols_keep_vt_]
            y_sample = y_series.iloc[sample_idx]
        else:
            X_sample = X_imp_df[self.cols_keep_vt_]
            y_sample = y_series

        selector_model = XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            tree_method='hist',
            random_state=self.random_state,
            n_jobs=-1,
            scale_pos_weight=1.0,
        )
        selector_model.fit(X_sample, y_sample)

        importances = pd.Series(selector_model.feature_importances_, index=self.cols_keep_vt_)
        self.feature_importances_ = importances.sort_values(ascending=False)
        self.selected_features_ = self.feature_importances_.head(self.n_features).index.tolist()
        return self

    def transform(self, X):
        return X[[c for c in self.selected_features_ if c in X.columns]]

# Full Pipeline

In [9]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scorer = 'roc_auc'


def build_xgb_pipeline(n_estimators, max_depth, learning_rate,
                       subsample, colsample_bytree, min_child_weight,
                       reg_lambda, scale_pos_weight,
                       use_undersampling=False):
    preprocessing = [
        ('cleaning', CleaningTransformer()),
        ('engineering', FeatureEngineeringTransformer()),
        ('selection', XGBFeatureSelectionTransformer(random_state=RANDOM_STATE)),
        ('imputer', SimpleImputer(strategy='median')),
    ]

    xgb = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        reg_lambda=reg_lambda,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    if use_undersampling:
        return ImbPipeline(preprocessing + [
            ('under', RandomUnderSampler(random_state=RANDOM_STATE)),
            ('xgb', xgb),
        ])

    return Pipeline(preprocessing + [('xgb', xgb)])

In [10]:
with mlflow.start_run(run_id=PARENT_RUN_ID):
    with mlflow.start_run(run_name='XGBoost_Cleaning', nested=True):
        mlflow.log_params({
            'stage': 'cleaning',
            'missing_threshold': cleaning_probe.missing_thresh,
            'dropped_missing_columns': len(cleaning_probe.cols_to_drop_),
            'kept_columns_after_cleaning': len(cleaning_probe.common_cols_),
            'dropped_id_column': ID_COL,
        })

    with mlflow.start_run(run_name='XGBoost_Feature_Engineering', nested=True):
        mlflow.log_params({
            'stage': 'feature_engineering',
            'amount_features': 'TransactionAmt_log,TransactionAmt_cents,Amt_is_round',
            'time_features': 'hour,day_of_week,is_weekend',
            'categorical_encoding': 'OrdinalEncoder(handle_unknown=use_encoded_value)',
            'categorical_columns': len(fe_probe.cat_cols_),
            'missing_indicator_columns': len(fe_probe.missing_flag_cols_),
            'scaling': 'not_used_for_xgboost',
        })

    with mlflow.start_run(run_name='XGBoost_Feature_Selection', nested=True):
        mlflow.log_params({
            'stage': 'feature_selection',
            'correlation_threshold': 0.98,
            'variance_threshold': 0.0,
            'selection_method': 'XGBClassifier_feature_importance',
            'selected_features': 80,
            'selector_sample_size': 80000,
        })

print('XGBoost preprocessing stage runs logged.')

🏃 View run XGBoost_Cleaning at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/a0d834dc90fd4fca9e13823959bf56c9
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGBoost_Feature_Engineering at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/ae3dd739df5446569e768b71281784c6
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/338bbcd5b9014c8f832ddadfb6d1033e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-

# Training

In [11]:
def evaluate_time_holdout(pipe, holdout_frac=0.20):
    from sklearn.base import clone

    cutoff = X_raw_train[TIME_COL].quantile(1 - holdout_frac)
    train_mask = X_raw_train[TIME_COL] <= cutoff
    val_mask = X_raw_train[TIME_COL] > cutoff

    time_pipe = clone(pipe)
    time_pipe.fit(X_raw_train.loc[train_mask], y_train.loc[train_mask])
    val_pred = time_pipe.predict_proba(X_raw_train.loc[val_mask])[:, 1]
    return roc_auc_score(y_train.loc[val_mask], val_pred)

def run_xgb_experiment(run_name, n_estimators, max_depth, learning_rate,
                       subsample=0.8, colsample_bytree=0.8,
                       min_child_weight=1, reg_lambda=1.0,
                       scale_pos_weight=BASE_SCALE_POS_WEIGHT,
                       use_undersampling=False):
    with mlflow.start_run(run_id=PARENT_RUN_ID):
        with mlflow.start_run(run_name=run_name, nested=True) as run:
            pipe = build_xgb_pipeline(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                min_child_weight=min_child_weight,
                reg_lambda=reg_lambda,
                scale_pos_weight=scale_pos_weight,
                use_undersampling=use_undersampling,
            )

            cv_results = cross_validate(
                pipe, X_raw_train, y_train,
                cv=CV, scoring=scorer,
                return_train_score=True,
                n_jobs=1,
                error_score='raise',
            )

            mean_val_auc = cv_results['test_score'].mean()
            std_val_auc = cv_results['test_score'].std()
            mean_train_auc = cv_results['train_score'].mean()
            overfit_gap = mean_train_auc - mean_val_auc
            time_holdout_auc = evaluate_time_holdout(pipe)

            mlflow.log_params({
                'n_estimators': n_estimators,
                'max_depth': max_depth,
                'learning_rate': learning_rate,
                'subsample': subsample,
                'colsample_bytree': colsample_bytree,
                'min_child_weight': min_child_weight,
                'reg_lambda': reg_lambda,
                'scale_pos_weight': scale_pos_weight,
                'use_undersampling': use_undersampling,
                'cv_folds': 5,
            })
            mlflow.log_metrics({
                'cv_val_roc_auc_mean': round(mean_val_auc, 5),
                'cv_val_roc_auc_std': round(std_val_auc, 5),
                'cv_train_roc_auc_mean': round(mean_train_auc, 5),
                'overfit_gap': round(overfit_gap, 5),
                'time_holdout_roc_auc': round(time_holdout_auc, 5),
            })
            for i, (tr_s, val_s) in enumerate(zip(
                cv_results['train_score'], cv_results['test_score']
            )):
                mlflow.log_metric(f'fold_{i+1}_train_auc', round(tr_s, 5))
                mlflow.log_metric(f'fold_{i+1}_val_auc', round(val_s, 5))

            print(f'{run_name}: val_AUC={mean_val_auc:.5f} ± {std_val_auc:.5f} | '
                  f'train_AUC={mean_train_auc:.5f} | gap={overfit_gap:.5f} | '
                  f'time_holdout_AUC={time_holdout_auc:.5f}')

            pipe.fit(X_raw_train, y_train)
            mlflow.sklearn.log_model(pipe, name='model')

            return mean_val_auc, std_val_auc, pipe, run.info.run_id

In [12]:
# Experiment 1: baseline weighted XGBoost
auc1, std1, pipe1, rid1 = run_xgb_experiment(
    run_name='XGB_01_Baseline_Weighted',
    n_estimators=300,
    max_depth=4,
    learning_rate=0.08,
    scale_pos_weight=BASE_SCALE_POS_WEIGHT,
)

XGB_01_Baseline_Weighted: val_AUC=0.91657 ± 0.00231 | train_AUC=0.92710 | gap=0.01054 | time_holdout_AUC=0.89518


2026/05/04 20:18:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGB_01_Baseline_Weighted at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/4bcc176f0cf147ed9efdfc14e7df7f96
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3


In [13]:
# Experiment 2: shallower model, underfitting check
auc2, std2, pipe2, rid2 = run_xgb_experiment(
    run_name='XGB_02_Shallow_Underfit',
    n_estimators=150,
    max_depth=2,
    learning_rate=0.05,
    min_child_weight=5,
    reg_lambda=5.0,
    scale_pos_weight=BASE_SCALE_POS_WEIGHT,
)

XGB_02_Shallow_Underfit: val_AUC=0.86484 ± 0.00258 | train_AUC=0.86607 | gap=0.00124 | time_holdout_AUC=0.85173


2026/05/04 20:32:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGB_02_Shallow_Underfit at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/6d7871be31024b979e283d584938ef54
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3


In [14]:
# Experiment 3: deeper model, overfitting check
auc3, std3, pipe3, rid3 = run_xgb_experiment(
    run_name='XGB_03_Deeper_Overfit_Check',
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    min_child_weight=1,
    reg_lambda=1.0,
    scale_pos_weight=BASE_SCALE_POS_WEIGHT,
)

XGB_03_Deeper_Overfit_Check: val_AUC=0.95410 ± 0.00085 | train_AUC=0.98684 | gap=0.03274 | time_holdout_AUC=0.89670


2026/05/04 20:49:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGB_03_Deeper_Overfit_Check at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/a9147ecbb4da4d4c963a4a0680cc96ea
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3


In [15]:
# Experiment 4: undersampling comparison
auc4, std4, pipe4, rid4 = run_xgb_experiment(
    run_name='XGB_04_UnderSampling',
    n_estimators=300,
    max_depth=4,
    learning_rate=0.08,
    scale_pos_weight=1.0,
    use_undersampling=True,
)

XGB_04_UnderSampling: val_AUC=0.91270 ± 0.00217 | train_AUC=0.92190 | gap=0.00920 | time_holdout_AUC=0.89327


2026/05/04 21:03:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGB_04_UnderSampling at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/dc30fc87298a49a5bfbf998a92cb9864
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3


In [16]:
# Experiment 5: stronger regularization
auc5, std5, pipe5, rid5 = run_xgb_experiment(
    run_name='XGB_05_Regularized',
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=10,
    reg_lambda=10.0,
    scale_pos_weight=BASE_SCALE_POS_WEIGHT,
)

XGB_05_Regularized: val_AUC=0.91284 ± 0.00191 | train_AUC=0.92172 | gap=0.00888 | time_holdout_AUC=0.89586


2026/05/04 21:18:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGB_05_Regularized at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/77d34836c670476e8a86a1a466224c37
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3


# Results

In [17]:
results = [
    {'run': 'XGB_01_Baseline_Weighted',   'val_auc': auc1, 'std': std1, 'run_id': rid1},
    {'run': 'XGB_02_Shallow_Underfit',    'val_auc': auc2, 'std': std2, 'run_id': rid2},
    {'run': 'XGB_03_Deeper_Overfit_Check','val_auc': auc3, 'std': std3, 'run_id': rid3},
    {'run': 'XGB_04_UnderSampling',       'val_auc': auc4, 'std': std4, 'run_id': rid4},
    {'run': 'XGB_05_Regularized',         'val_auc': auc5, 'std': std5, 'run_id': rid5},
]

results_df = pd.DataFrame(results).sort_values('val_auc', ascending=False)
print(results_df[['run', 'val_auc', 'std']].to_string(index=False))

best_row = results_df.iloc[0]
BEST_PIPE_NAME = best_row['run']
BEST_RUN_ID = best_row['run_id']
print(f'Best XGBoost model: {BEST_PIPE_NAME} (val AUC = {best_row["val_auc"]:.5f})')

                        run  val_auc      std
XGB_03_Deeper_Overfit_Check 0.954101 0.000849
   XGB_01_Baseline_Weighted 0.916566 0.002314
         XGB_05_Regularized 0.912838 0.001908
       XGB_04_UnderSampling 0.912699 0.002167
    XGB_02_Shallow_Underfit 0.864835 0.002575
Best XGBoost model: XGB_03_Deeper_Overfit_Check (val AUC = 0.95410)


In [18]:
pipeline_map = {
    'XGB_01_Baseline_Weighted': pipe1,
    'XGB_02_Shallow_Underfit': pipe2,
    'XGB_03_Deeper_Overfit_Check': pipe3,
    'XGB_04_UnderSampling': pipe4,
    'XGB_05_Regularized': pipe5,
}

best_pipeline = pipeline_map[BEST_PIPE_NAME]

with mlflow.start_run(run_id=PARENT_RUN_ID):
    mlflow.log_metric('best_val_roc_auc', round(best_row['val_auc'], 5))
    mlflow.log_param('best_child_run', BEST_PIPE_NAME)
    mlflow.log_param('best_run_id', BEST_RUN_ID)

print('Best XGBoost pipeline selected:', BEST_PIPE_NAME)
print(best_pipeline)

🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
Best XGBoost pipeline selected: XGB_03_Deeper_Overfit_Check
Pipeline(steps=[('cleaning', CleaningTransformer()),
                ('engineering', FeatureEngineeringTransformer()),
                ('selection', XGBFeatureSelectionTransformer()),
                ('imputer', SimpleImputer(strategy='median')),
                ('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_r...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
            

In [20]:
MODEL_REGISTRY_NAME = 'XGB_BestPipeline'

with mlflow.start_run(run_id=PARENT_RUN_ID):
    with mlflow.start_run(run_name='XGBoost_Best_Model_Save', nested=True):
        mlflow.log_param('saved_pipeline', BEST_PIPE_NAME)
        mlflow.log_metric('best_val_roc_auc', round(best_row['val_auc'], 5))

        input_example = X_raw_train.head(5)
        model_info = mlflow.sklearn.log_model(
            sk_model=best_pipeline,
            name='best_xgb_pipeline',
            input_example=input_example,
            registered_model_name='XGB_BestPipeline',
        )
        print(f'Best XGBoost pipeline saved as: {MODEL_REGISTRY_NAME}')
        print(f'Model URI: {model_info.model_uri}')

2026/05/04 21:38:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_BestPipeline'.
2026/05/04 21:38:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_BestPipeline, version 1
Created version '1' of model 'XGB_BestPipeline'.


Best XGBoost pipeline saved as: XGB_BestPipeline
Model URI: models:/m-aa5cadf735864649b35caf5519571485
🏃 View run XGBoost_Best_Model_Save at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/4bad4b849a674a69b88e219e442197fb
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
🏃 View run XGB_Parent at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3/runs/191ca1f68a9e404eaea3a0f32d9f031e
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/3
